# Planner Evaluation

Compare model performance on tool selection tasks for retail customer service.

## Models compared

| Model | Description |
|-------|-------------|
| **o4-mini (vanilla)** | Small reasoning model, no fine-tuning |
| **gpt-5.2** (none/low/medium/high) | Large model with varying reasoning effort |
| **o4-mini (fine-tuned)** | Small model with reinforcement fine-tuning (RFT) |

## Metrics (aligned with training grader)

| Metric | Description |
|--------|-------------|
| **Recall** | % of expected tools found in response |
| **Precision** | % of predicted tools that were expected |
| **F2 Score** | Recall-weighted F-score: `5 * (P * R) / (4 * P + R)` |

F2 (β=2) prioritizes recall over precision because missing a tool causes workflow failure, while extra tools only add minor API cost.

This ensures evaluation is **directly comparable** to the RFT training reward signal.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.settings import (
    DATA_DIR, DATASETS_DIR,
    PLANNER_OUTPUTS_DIR, PLANNER_RESPONSES_DIR,
    AZURE_DEPLOYMENT, AZURE_DEPLOYMENT_VANILLA, FINETUNED_DEPLOYMENT,
    load_planner_schema
)

# Load structured output schema for consistent evaluation
PLANNER_SCHEMA = load_planner_schema()
print(f"📋 Structured output schema loaded: {PLANNER_SCHEMA['json_schema']['name']}")

---

## Load Evaluation Data

We use **validation + test sets** (30 samples total) for evaluation.

| Dataset | Samples | Usage during training |
|---------|---------|----------------------|
| `val.jsonl` | 20 | Reward calculation only (no gradient updates) |
| `test.jsonl` | 10 | Held out completely |

In [ ]:
from src.data_utils import load_jsonl

eval_samples = []
for filepath in [DATASETS_DIR / "val.jsonl", DATASETS_DIR / "test.jsonl"]:
    if filepath.exists():
        eval_samples.extend(load_jsonl(filepath))

print(f"📋 Loaded {len(eval_samples)} evaluation samples")

<cell_type>markdown</cell_type>---

## Evaluators

Custom evaluators **fully aligned with the training grader**:

| Evaluator | Logic (same as grader) |
|-----------|------------------------|
| **RecallEvaluator** | % of expected tools found in response |
| **PrecisionEvaluator** | `correct / predicted` — ratio of correct tools |
| **PlannerEvaluator** | F2 score: `5 * (precision * recall) / (4 * precision + recall)` |

In [ ]:
from src.evaluation.evaluators import PlannerEvaluator, test_evaluators

# Quick test
test_evaluators()

---

## Generate Responses

Generate responses from both models in parallel using async requests.

In [ ]:
from src.evaluation.generate import generate_responses_for_notebook

# =============================================================================
# CONFIGURATION
# =============================================================================
MAX_CONCURRENT = 5  # Adjust based on TPM limits
# =============================================================================

print("📊 Generating responses for 6 configurations (with structured output)...")

# 1. o4-mini (vanilla) - reasoning model
baseline_file, baseline_usage = await generate_responses_for_notebook(
    AZURE_DEPLOYMENT, eval_samples, MAX_CONCURRENT,
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ o4-mini (vanilla): {baseline_file}")

# 2. gpt-5.2 (reasoning: none)
vanilla_none_file, vanilla_none_usage = await generate_responses_for_notebook(
    AZURE_DEPLOYMENT_VANILLA, eval_samples, MAX_CONCURRENT,
    reasoning_effort="none", output_suffix="none",
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ gpt-5.2 (none): {vanilla_none_file}")

# 3. gpt-5.2 (reasoning: low)
vanilla_low_file, vanilla_low_usage = await generate_responses_for_notebook(
    AZURE_DEPLOYMENT_VANILLA, eval_samples, MAX_CONCURRENT,
    reasoning_effort="low", output_suffix="low",
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ gpt-5.2 (low): {vanilla_low_file}")

# 4. gpt-5.2 (reasoning: medium)
vanilla_medium_file, vanilla_medium_usage = await generate_responses_for_notebook(
    AZURE_DEPLOYMENT_VANILLA, eval_samples, MAX_CONCURRENT,
    reasoning_effort="medium", output_suffix="medium",
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ gpt-5.2 (medium): {vanilla_medium_file}")

# 5. gpt-5.2 (reasoning: high)
vanilla_high_file, vanilla_high_usage = await generate_responses_for_notebook(
    AZURE_DEPLOYMENT_VANILLA, eval_samples, MAX_CONCURRENT,
    reasoning_effort="high", output_suffix="high",
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ gpt-5.2 (high): {vanilla_high_file}")

# 6. o4-mini (fine-tuned) - RFT trained
finetuned_file, finetuned_usage = await generate_responses_for_notebook(
    FINETUNED_DEPLOYMENT, eval_samples, MAX_CONCURRENT,
    response_format=PLANNER_SCHEMA
)
print(f"   ✅ o4-mini (fine-tuned): {finetuned_file}")

# Store all usages for cost analysis
usage_data = {
    "o4-mini (vanilla)": baseline_usage,
    "gpt-5.2 (none)": vanilla_none_usage,
    "gpt-5.2 (low)": vanilla_low_usage,
    "gpt-5.2 (medium)": vanilla_medium_usage,
    "gpt-5.2 (high)": vanilla_high_usage,
    "o4-mini (fine-tuned)": finetuned_usage,
}

print("\n📈 Token Usage Summary:")
print("-" * 90)
print(f"{'Model':<22} {'Input':>10} {'Output':>10} {'Reasoning':>12} {'Total':>10}")
print("-" * 90)
for name, usage in usage_data.items():
    print(f"{name:<22} {usage['input_tokens']:>10,} {usage['output_tokens']:>10,} {usage['reasoning_tokens']:>12,} {usage['total_tokens']:>10,}")
print("-" * 90)

---

## Run Evaluation

In [ ]:
from azure.ai.evaluation import evaluate
from src.evaluation.evaluators import PlannerEvalWrapper

print("🔬 Evaluating 6 configurations...")

print("   1/6 o4-mini (vanilla)...")
baseline_results = evaluate(data=baseline_file, evaluators={"planner": PlannerEvalWrapper()})

print("   2/6 gpt-5.2 (none)...")
vanilla_none_results = evaluate(data=vanilla_none_file, evaluators={"planner": PlannerEvalWrapper()})

print("   3/6 gpt-5.2 (low)...")
vanilla_low_results = evaluate(data=vanilla_low_file, evaluators={"planner": PlannerEvalWrapper()})

print("   4/6 gpt-5.2 (medium)...")
vanilla_medium_results = evaluate(data=vanilla_medium_file, evaluators={"planner": PlannerEvalWrapper()})

print("   5/6 gpt-5.2 (high)...")
vanilla_high_results = evaluate(data=vanilla_high_file, evaluators={"planner": PlannerEvalWrapper()})

print("   6/6 o4-mini (fine-tuned)...")
finetuned_results = evaluate(data=finetuned_file, evaluators={"planner": PlannerEvalWrapper()})

print("\n✅ All evaluations complete!")

---

## Results Comparison

In [ ]:
import pandas as pd

# Load result DataFrames
baseline_df = pd.DataFrame(baseline_results["rows"])
vanilla_none_df = pd.DataFrame(vanilla_none_results["rows"])
vanilla_low_df = pd.DataFrame(vanilla_low_results["rows"])
vanilla_medium_df = pd.DataFrame(vanilla_medium_results["rows"])
vanilla_high_df = pd.DataFrame(vanilla_high_results["rows"])
finetuned_df = pd.DataFrame(finetuned_results["rows"])

# Column mapping
col_mapping = {
    "recall": "outputs.planner.recall",
    "precision": "outputs.planner.precision",
    "f2": "outputs.planner.f2"
}

# Build results dict for easy access
all_results = {
    "o4-mini (vanilla)": baseline_df,
    "gpt-5.2 (none)": vanilla_none_df,
    "gpt-5.2 (low)": vanilla_low_df,
    "gpt-5.2 (medium)": vanilla_medium_df,
    "gpt-5.2 (high)": vanilla_high_df,
    "o4-mini (fine-tuned)": finetuned_df
}

# Calculate F2 scores (used in cost analysis and visualization)
f2_scores = {name: df[col_mapping['f2']].mean() for name, df in all_results.items()}

# Print comparison table
print("\n" + "=" * 115)
print(f"{'Metric':<10}", end="")
for name in all_results.keys():
    print(f"{name:>18}", end="")
print(f"{'Best':>20}")
print("-" * 115)

for metric_name, col in col_mapping.items():
    print(f"{metric_name:<10}", end="")
    scores = {}
    for name, df in all_results.items():
        score = df[col].mean()
        scores[name] = score
        print(f"{score:>18.3f}", end="")
    best = max(scores, key=scores.get)
    print(f"{best:>20}")

print("=" * 115)

---

## Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

metrics = ['Recall', 'Precision', 'F2']
model_names = list(all_results.keys())
colors = ['#3498db', '#9b59b6', '#e74c3c', '#f39c12', '#1abc9c', '#2ecc71']

# Get scores for each model
scores_by_model = []
for name, df in all_results.items():
    scores_by_model.append([df[col_mapping[m.lower()]].mean() for m in metrics])

x = range(len(metrics))
width = 0.12
offsets = [i - 2.5*width for i in range(len(model_names))]

for i, (name, scores, color) in enumerate(zip(model_names, scores_by_model, colors)):
    offset = (i - 2.5) * width
    bars = ax.bar([xi + offset for xi in x], scores, width, label=name, color=color)
    # Add values on bars
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.2f}', ha='center', va='bottom', fontsize=6, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(loc='lower right', fontsize=8)
ax.set_ylim(0.5, 1.1)
ax.set_ylabel('Score')
ax.set_title('RFT Planner Evaluation: o4-mini (vanilla) vs gpt-5.2 (all reasoning levels) vs o4-mini (fine-tuned)')

plt.tight_layout()
plt.savefig(PLANNER_OUTPUTS_DIR / 'eval_results.png', dpi=150)
plt.show()

print(f"\n📊 Saved: {PLANNER_OUTPUTS_DIR / 'eval_results.png'}")

---

## Export Results

In [ ]:
import json
from datetime import datetime
from IPython.display import display, Markdown

# Save all CSVs
baseline_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_o4mini_vanilla.csv', index=False)
vanilla_none_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_gpt52_none.csv', index=False)
vanilla_low_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_gpt52_low.csv', index=False)
vanilla_medium_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_gpt52_medium.csv', index=False)
vanilla_high_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_gpt52_high.csv', index=False)
finetuned_df.to_csv(PLANNER_OUTPUTS_DIR / 'eval_o4mini_finetuned.csv', index=False)

# Save summary JSON (with token usage)
summary = {
    "timestamp": datetime.now().isoformat(),
    "models": {
        "o4-mini (vanilla)": AZURE_DEPLOYMENT,
        "gpt-5.2 (none)": f"{AZURE_DEPLOYMENT_VANILLA} (reasoning: none)",
        "gpt-5.2 (low)": f"{AZURE_DEPLOYMENT_VANILLA} (reasoning: low)",
        "gpt-5.2 (medium)": f"{AZURE_DEPLOYMENT_VANILLA} (reasoning: medium)",
        "gpt-5.2 (high)": f"{AZURE_DEPLOYMENT_VANILLA} (reasoning: high)",
        "o4-mini (fine-tuned)": FINETUNED_DEPLOYMENT
    },
    "samples": len(eval_samples),
    "results": {
        name: {metric: float(df[col].mean()) for metric, col in col_mapping.items()}
        for name, df in all_results.items()
    },
    "token_usage": {
        name: {
            "input_tokens": usage["input_tokens"],
            "output_tokens": usage["output_tokens"],
            "reasoning_tokens": usage["reasoning_tokens"],
            "total_tokens": usage["total_tokens"]
        }
        for name, usage in usage_data.items()
    }
}

with open(PLANNER_OUTPUTS_DIR / 'eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Display with Markdown
md = f"""### 💾 Saved to `{PLANNER_OUTPUTS_DIR}`

**CSV files:**
- `eval_o4mini_vanilla.csv`
- `eval_gpt52_none.csv`
- `eval_gpt52_low.csv`
- `eval_gpt52_medium.csv`
- `eval_gpt52_high.csv`
- `eval_o4mini_finetuned.csv`

**Summary:** `eval_summary.json` (with token usage)

**Chart:** `eval_results.png`
"""

display(Markdown(md))

---

## Cost Analysis

Estimate inference costs based on Azure OpenAI pricing (East US 2 - Global Standard, December 2024).

| Model | Input | Output | Notes |
|-------|-------|--------|-------|
| **o4-mini** | $1.10/1M | $4.40/1M | Reasoning tokens included in output |
| **gpt-5.2** | $1.75/1M | $14.00/1M | Reasoning tokens billed at output rate |
| **gpt-5-mini** | $0.25/1M | $2.00/1M | For reference |

**Fine-tuning costs** (one-time):
- Training: ~$100/hour for o4-mini RFT
- Hosting: ~$1.70/hour for Standard deployment (or $0 for Developer tier)

In [ ]:
# =============================================================================
# COST ANALYSIS (Azure East US 2 - Global Standard, December 2025)
# =============================================================================
from IPython.display import display, Markdown
from src.cost import calculate_model_cost, calculate_breakeven

# =============================================================================
# CONFIGURATION - Adjust these values based on your actual costs
# =============================================================================
# Pricing ($/1M tokens)
O4_MINI_INPUT = 1.10
O4_MINI_OUTPUT = 4.40
GPT52_INPUT = 1.75
GPT52_OUTPUT = 14.00

# Fine-tuning costs
TRAINING_RUNS = 1              # Number of training experiments before success
TRAINING_HOURS_PER_RUN = 2.0   # Hours per training run
TRAINING_RATE = 100.0          # $/hour for o4-mini RFT
HOSTING_RATE = 1.70            # $/hour for Standard deployment (0 for Developer)
HOSTING_HOURS_MONTHLY = 720    # 30 days always-on

# Projection
MONTHLY_REQUESTS = 300000     # Projected monthly request volume
PROJECTION_MONTHS = 12          # Project costs over X months

# Amortization
AMORTIZATION_MONTHS = 6        # Spread training costs over X months
# =============================================================================

N_SAMPLES = len(eval_samples)

# Calculate costs for each model
cost_results = {}

# o4-mini (vanilla) - pay-per-use, no hosting
cost_results["o4-mini (vanilla)"] = calculate_model_cost(
    usage=usage_data["o4-mini (vanilla)"],
    input_price=O4_MINI_INPUT,
    output_price=O4_MINI_OUTPUT,
    num_samples=N_SAMPLES,
    training_runs=0,
    hosting_hours_monthly=0,
    monthly_requests=MONTHLY_REQUESTS
)

# gpt-5.2 variants (pay-per-use, no hosting)
for level in ["none", "low", "medium", "high"]:
    cost_results[f"gpt-5.2 ({level})"] = calculate_model_cost(
        usage=usage_data[f"gpt-5.2 ({level})"],
        input_price=GPT52_INPUT,
        output_price=GPT52_OUTPUT,
        num_samples=N_SAMPLES,
        training_runs=0,
        hosting_hours_monthly=0,
        monthly_requests=MONTHLY_REQUESTS
    )

# o4-mini (fine-tuned) - with training/hosting costs
cost_results["o4-mini (fine-tuned)"] = calculate_model_cost(
    usage=usage_data["o4-mini (fine-tuned)"],
    input_price=O4_MINI_INPUT,
    output_price=O4_MINI_OUTPUT,
    num_samples=N_SAMPLES,
    training_runs=TRAINING_RUNS,
    training_hours_per_run=TRAINING_HOURS_PER_RUN,
    training_rate=TRAINING_RATE,
    hosting_hours_monthly=HOSTING_HOURS_MONTHLY,
    hosting_rate=HOSTING_RATE,
    amortization_months=AMORTIZATION_MONTHS,
    monthly_requests=MONTHLY_REQUESTS
)

# Build display table
md = f"""### 💰 COST ANALYSIS

**Configuration:**
- Training: {TRAINING_RUNS} run × {TRAINING_HOURS_PER_RUN}h @ ${TRAINING_RATE}/h = **${TRAINING_RUNS * TRAINING_HOURS_PER_RUN * TRAINING_RATE:.0f}** (one-time)
- Hosting: {HOSTING_HOURS_MONTHLY}h/month @ ${HOSTING_RATE}/h = **${HOSTING_HOURS_MONTHLY * HOSTING_RATE:,.0f}**/month
- Projection: **{MONTHLY_REQUESTS:,} requests/month** over **{PROJECTION_MONTHS} months**

| Model | Reasoning | Total Tokens | Cost/100k req | Training | Hosting/mo | **{PROJECTION_MONTHS}-mo Total** |
|-------|----------:|-------------:|--------------:|---------:|-----------:|------------------:|
"""

for name in ["o4-mini (vanilla)", "gpt-5.2 (none)", "gpt-5.2 (low)", "gpt-5.2 (medium)", "gpt-5.2 (high)", "o4-mini (fine-tuned)"]:
    usage = usage_data[name]
    cost = cost_results[name]
    
    reasoning = usage["reasoning_tokens"]
    total = usage["total_tokens"]
    
    # Cost for 100k requests = cost_per_1k * 100
    cost_100k = cost["inference_cost_per_1k"] * 100
    
    # Training (one-time)
    training = cost["training_cost_total"]
    training_str = f"${training:,.0f}" if training > 0 else "-"
    
    # Hosting per month
    hosting = cost["hosting_cost_monthly"]
    hosting_str = f"${hosting:,.0f}" if hosting > 0 else "-"
    
    # N-month projection: training + (hosting + inference) * N
    inference_monthly = cost["inference_cost_per_1k"] * (MONTHLY_REQUESTS / 1000)
    total_projection = training + (hosting + inference_monthly) * PROJECTION_MONTHS
    
    md += f"| {name} | {reasoning:,} | {total:,} | ${cost_100k:,.0f} | {training_str} | {hosting_str} | **${total_projection:,.0f}** |\n"

# Store for later use
baseline_cost_per_1k = cost_results["o4-mini (vanilla)"]["inference_cost_per_1k"]
finetuned_cost_per_1k = cost_results["o4-mini (fine-tuned)"]["inference_cost_per_1k"]
gpt52_high_cost_per_1k = cost_results["gpt-5.2 (high)"]["inference_cost_per_1k"]

# Summary insights
baseline_total = cost_results["o4-mini (vanilla)"]["inference_cost_per_1k"] * (MONTHLY_REQUESTS / 1000) * PROJECTION_MONTHS
finetuned_total = cost_results["o4-mini (fine-tuned)"]["training_cost_total"] + \
    (cost_results["o4-mini (fine-tuned)"]["hosting_cost_monthly"] + \
     cost_results["o4-mini (fine-tuned)"]["inference_cost_per_1k"] * (MONTHLY_REQUESTS / 1000)) * PROJECTION_MONTHS
gpt52_high_total = cost_results["gpt-5.2 (high)"]["inference_cost_per_1k"] * (MONTHLY_REQUESTS / 1000) * PROJECTION_MONTHS

md += f"""
**📊 {PROJECTION_MONTHS}-Month Cost Comparison @ {MONTHLY_REQUESTS:,} requests/month:**
- o4-mini (vanilla): **${baseline_total:,.0f}**
- o4-mini (fine-tuned): **${finetuned_total:,.0f}** (includes ${cost_results["o4-mini (fine-tuned)"]["training_cost_total"]:.0f} training + ${cost_results["o4-mini (fine-tuned)"]["hosting_cost_monthly"]:,.0f}/mo hosting)
- gpt-5.2 (high): **${gpt52_high_total:,.0f}**
"""

display(Markdown(md))

In [ ]:
# =============================================================================
# ROI ANALYSIS: o4-mini (fine-tuned) vs gpt-5.2 high (closest in performance)
# =============================================================================
from IPython.display import display, Markdown

# Get costs from cell-17
training_cost_total = cost_results["o4-mini (fine-tuned)"]["training_cost_total"]
hosting_cost_monthly = cost_results["o4-mini (fine-tuned)"]["hosting_cost_monthly"]

md = f"""### 📈 ROI ANALYSIS: o4-mini (fine-tuned) vs gpt-5.2 (high)

**Why gpt-5.2 (high)?** Closest in F2 performance:
- o4-mini (fine-tuned): F2 = {f2_scores["o4-mini (fine-tuned)"]:.3f}
- gpt-5.2 (high): F2 = {f2_scores["gpt-5.2 (high)"]:.3f}

**💵 Fine-tuning Investment:**
- Training: {TRAINING_RUNS} runs × {TRAINING_HOURS_PER_RUN}h × ${TRAINING_RATE}/h = **${training_cost_total:,.0f}** (one-time)
- Hosting: **${hosting_cost_monthly:,.0f}**/month

**📊 Inference Cost Comparison (per 1,000 requests):**
| Model | Cost/1k req |
|-------|------------:|
| o4-mini (fine-tuned) | ${finetuned_cost_per_1k:.2f} |
| gpt-5.2 (high) | ${gpt52_high_cost_per_1k:.2f} |
| **Inference savings** | **${gpt52_high_cost_per_1k - finetuned_cost_per_1k:.2f}** |
"""

# Break-even calculation
breakeven_high = calculate_breakeven(
    finetuned_cost_per_1k=finetuned_cost_per_1k,
    alternative_cost_per_1k=gpt52_high_cost_per_1k,
    training_cost_total=training_cost_total,
    hosting_cost_monthly=hosting_cost_monthly,
    amortization_months=AMORTIZATION_MONTHS
)

if breakeven_high["is_viable"]:
    md += f"""
**🎯 Break-even Analysis:**

To cover the **${hosting_cost_monthly:,.0f}/month hosting cost**, you need enough requests 
so that inference savings = hosting cost:

```
break_even = hosting_monthly / savings_per_1k × 1000
           = ${hosting_cost_monthly:,.0f} / ${breakeven_high['savings_per_1k']:.2f} × 1000
           = {breakeven_high['breakeven_monthly']:,} requests/month
           = {breakeven_high['breakeven_daily']:,} requests/day
```

**💡 At your projected volume ({MONTHLY_REQUESTS:,} requests/month):**
"""
    # Correct calculation: net = inference_savings - hosting
    inference_savings_monthly = breakeven_high['savings_per_1k'] * (MONTHLY_REQUESTS / 1000)
    net_monthly = inference_savings_monthly - hosting_cost_monthly
    
    md += f"""
| Component | Amount |
|-----------|-------:|
| Inference savings | +${inference_savings_monthly:,.0f} |
| Hosting cost | -${hosting_cost_monthly:,.0f} |
| **Net monthly** | **${net_monthly:+,.0f}** |
"""
    
    if net_monthly > 0:
        md += f"\n✅ **Above break-even!** You save ${net_monthly:,.0f}/month vs gpt-5.2 (high)."
    else:
        md += f"\n⚠️ **Below break-even.** You lose ${-net_monthly:,.0f}/month vs gpt-5.2 (high). Need {breakeven_high['breakeven_monthly']:,} req/month to break even."

else:
    md += f"""
**⚠️ o4-mini (fine-tuned) is MORE expensive per request than gpt-5.2 (high)!**
- Additional cost: ${-breakeven_high['savings_per_1k']:.2f} per 1,000 requests
- Recommendation: Use gpt-5.2 (high) directly, or try Developer tier (no hosting cost).
"""

# Total cost projection
finetuned_monthly_total = hosting_cost_monthly + (finetuned_cost_per_1k * MONTHLY_REQUESTS / 1000)
gpt52_high_monthly = gpt52_high_cost_per_1k * MONTHLY_REQUESTS / 1000

finetuned_total = training_cost_total + (finetuned_monthly_total * PROJECTION_MONTHS)
gpt52_high_total = gpt52_high_monthly * PROJECTION_MONTHS

diff_total = gpt52_high_total - finetuned_total

md += f"""
---
**📊 {PROJECTION_MONTHS}-Month Total Cost @ {MONTHLY_REQUESTS:,} requests/month:**

| Model | Monthly | {PROJECTION_MONTHS}-Month Total |
|-------|--------:|----------------:|
| o4-mini (fine-tuned) | ${finetuned_monthly_total:,.0f} | ${finetuned_total:,.0f} |
| gpt-5.2 (high) | ${gpt52_high_monthly:,.0f} | ${gpt52_high_total:,.0f} |
| **Difference** | | **${diff_total:+,.0f}** |

*o4-mini (fine-tuned) total = ${training_cost_total:,.0f} training + ({PROJECTION_MONTHS} × ${finetuned_monthly_total:,.0f})*
"""

display(Markdown(md))

In [ ]:
# =============================================================================
# VISUALIZATION: Cost vs Performance Trade-off
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = ["o4-mini (vanilla)", "gpt-5.2 (none)", "gpt-5.2 (low)", "gpt-5.2 (medium)", "gpt-5.2 (high)", "o4-mini (fine-tuned)"]
colors = ['#3498db', '#9b59b6', '#e74c3c', '#f39c12', '#1abc9c', '#2ecc71']

# Get costs from cost_results
costs = [cost_results[name]["inference_cost_per_1k"] for name in model_names]

# Chart 1: Cost comparison (bar chart)
ax1 = axes[0]
bars = ax1.bar(model_names, costs, color=colors)
ax1.set_ylabel('Cost per 1,000 requests (USD)')
ax1.set_title('Inference Cost Comparison')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar, cost in zip(bars, costs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'${cost:.2f}', ha='center', va='bottom', fontsize=9)

# Chart 2: F2 vs Cost scatter plot
ax2 = axes[1]
for i, name in enumerate(model_names):
    f2 = f2_scores[name]
    cost = cost_results[name]["inference_cost_per_1k"]
    ax2.scatter(cost, f2, s=150, c=colors[i], label=name, edgecolors='black', linewidth=1)
    ax2.annotate(name, (cost, f2), textcoords="offset points", 
                 xytext=(5, 5), fontsize=8, ha='left')

ax2.set_xlabel('Cost per 1,000 requests (USD)')
ax2.set_ylabel('F2 Score')
ax2.set_title('Performance vs Cost Trade-off')
ax2.grid(True, alpha=0.3)

# Highlight Fine-tuned position
ax2.axhline(y=f2_scores["o4-mini (fine-tuned)"], color='green', linestyle='--', alpha=0.5)
ax2.axvline(x=cost_results["o4-mini (fine-tuned)"]["inference_cost_per_1k"], color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(PLANNER_OUTPUTS_DIR / 'cost_analysis.png', dpi=150)
plt.show()

print(f"\n📊 Saved: {PLANNER_OUTPUTS_DIR / 'cost_analysis.png'}")

---

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================
from IPython.display import display, Markdown

baseline_f2 = f2_scores["o4-mini (vanilla)"]

# Build summary
md = f"""### 🏆 EVALUATION COMPLETE

| Config | Model |
|--------|-------|
| o4-mini (vanilla) | {AZURE_DEPLOYMENT} |
| gpt-5.2 (none) | {AZURE_DEPLOYMENT_VANILLA} (reasoning: none) |
| gpt-5.2 (low) | {AZURE_DEPLOYMENT_VANILLA} (reasoning: low) |
| gpt-5.2 (medium) | {AZURE_DEPLOYMENT_VANILLA} (reasoning: medium) |
| gpt-5.2 (high) | {AZURE_DEPLOYMENT_VANILLA} (reasoning: high) |
| o4-mini (fine-tuned) | {FINETUNED_DEPLOYMENT} |

**Samples:** {len(eval_samples)}

**F2 Scores (vs o4-mini vanilla):**
| Model | F2 Score | Delta |
|-------|----------|-------|
"""

for name, f2 in f2_scores.items():
    delta = ((f2 - baseline_f2) / baseline_f2 * 100) if baseline_f2 > 0 else 0
    md += f"| {name} | {f2:.3f} | {delta:+.1f}% |\n"

# Determine winner
winner = max(f2_scores, key=f2_scores.get)
winner_score = f2_scores[winner]

if winner == "o4-mini (fine-tuned)":
    md += "\n🎉 **o4-mini (fine-tuned) wins!** RFT provides value even vs larger models with reasoning."
elif "5.2" in winner:
    md += f"\n⚠️ **{winner} wins** with F2={winner_score:.3f}. Consider cost/latency tradeoffs vs using a larger model with reasoning."
else:
    md += "\n🤔 **o4-mini (vanilla) wins.** Something may be wrong with the fine-tuning."

md += "\n\n🚀 Run `05_multiagent_with_tool_calling.ipynb` to test the multi-agent workflow!"

display(Markdown(md))